<div style="border-left:4px solid #a1a1aa;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#a1a1aa;">Setup</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Clones the code, downloads the models, builds the database and the index.</div></div>

**Setup** &nbsp;|&nbsp; [Understanding](https://www.kaggle.com/code/kirazul/nl2sql-2-understanding) &nbsp;|&nbsp; [Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures) &nbsp;|&nbsp; [Run All](https://www.kaggle.com/code/kirazul/nl2sql-4-run-all)

Run this notebook once, then **Save Version > Save & Run All**. The other three
attach its output and download nothing.

| Built here | Size | Used for |
|---|---|---|
| `eicu.db` | 429 MB | 31 tables, 4.6 M rows, with keys and indexes rebuilt |
| `value_index.db` | 2.4 MB | turning *aspirin* into the exact stored spelling |
| GLiNER2 | 806 MB | finding the entities in a question |
| Qwen3-1.7B Q4 | 1.1 GB | writing the answer, on the CPU |

First run takes 15 to 20 minutes, almost all of it transfer.

The data is **eICU-CRD v2.0.1**, a de-identified public research database of
intensive-care records published by the MIT Laboratory for Computational
Physiology. No private data is used anywhere in this project.

---

## 1. Environment

Two CPU cores, no GPU. Everything in this project is built to run on that.

In [ ]:
import os, sys, shutil
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
free_gb = shutil.disk_usage("/kaggle/working" if ON_KAGGLE else ".").free / 1e9

print(f"  platform    {'Kaggle' if ON_KAGGLE else 'workstation'}")
print(f"  python      {sys.version.split()[0]}")
print(f"  cores       {os.cpu_count()}")
print(f"  disk free   {free_gb:.1f} GB")

assert free_gb > 6, "6 GB of free disk is required"

---

## 2. The code

`hybridsql` is cloned from GitHub rather than pasted into cells, so what runs
here is what the tests were run against.

The repository is private. Add a GitHub personal access token with `repo` scope
as a secret named `GITHUB_TOKEN`, under **Add-ons > Secrets**.

```
src/hybridsql/
  db/          schema, read-only connection, value index
  pipeline/    understand > anonymize > generate > answer
  providers/   cloud (the only outbound socket), extractor, local model
  security/    egress gate, SQL validator, audit journal
  graph/       the four architectures, as state machines
  api/         the REST service
```

In [ ]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
INPUTS    = Path("/kaggle/input")
REPO      = "https://github.com/Kirazul/NL2SQL-demo.git"

SECRETS = {
    "GITHUB_TOKEN":       "clone the code (the repository is private)",
    "GROQ_API_KEY":       "the cloud model that writes the SQL",
    "OPENROUTER_API_KEY": "fallback when Groq rate-limits",
    "LANGSMITH_API_KEY":  "tracing backend",
    "PUBLISH_TOKEN":      "announce this session to the web interface",
}
REQUIRED = ()


WHY = {}          # label -> why it could not be read, when it could not


def secret(label, default=""):
    """One secret, by label. Kaggle grants access per notebook, not per account.

    The reason a lookup failed is kept rather than swallowed: "not attached to
    this notebook" and "the backend refused" both end as an empty string, and
    without the reason the two are indistinguishable from the output.
    """
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(label)
            if value:
                return value
            WHY[label] = "Kaggle returned an empty value"
        except Exception as error:
            WHY[label] = f"{type(error).__name__}: {str(error)[:110]}"
    return os.environ.get(label, default)


def load_secrets(project=None):
    """Read every label into the environment and print what was found.

    An empty secret is removed rather than set blank, so the package falls back to
    its own default instead of an empty string.
    """
    local = {}
    if not ON_KAGGLE and project and (project / ".env").exists():
        for line in (project / ".env").read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                label, _, value = line.partition("=")
                local[label.strip()] = value.strip().strip("\"'")

    for label in SECRETS:
        value = secret(label) or local.get(label, "")
        if value:
            os.environ[label] = value
        else:
            os.environ.pop(label, None)

    for label, purpose in SECRETS.items():
        if os.environ.get(label):
            state = "ok"
        elif label in REQUIRED:
            state = "REQUIRED"
        else:
            state = "-"
        print(f"  {label:<20}{state:<10}{purpose}")

    if WHY:
        print("\n  why a secret could not be read")
        for label, reason in WHY.items():
            print(f"    {label:<20}{reason}")

    absent = [l for l in REQUIRED if not os.environ.get(l)]
    if absent:
        where = "Add-ons > Secrets, in this notebook" if ON_KAGGLE else ".env"
        print(f"\n  Missing: {', '.join(absent)}. Set it in {where} and run this cell again.")
    return not absent


def get_code():
    """Clone the repository into a writable directory and put it on the path.

    Kaggle mounts every input read-only and notebook 1 writes a database next to
    the package, so the code never runs from where it is mounted.
    """
    if (Path.cwd() / "src/hybridsql").exists():
        return Path.cwd()

    target = WORK / "nl2sql"
    if (target / "src/hybridsql").exists():
        return target

    token = secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    done = subprocess.run(["git", "clone", "--depth", "1", "--quiet", url, str(target)],
                          capture_output=True, text=True)
    if done.returncode:
        detail = done.stderr.replace(token, "***") if token else done.stderr
        raise SystemExit(
            "Could not clone the repository.\n\n"
            "  It is private, so this notebook needs a GITHUB_TOKEN secret:\n"
            "  Add-ons > Secrets > attach GITHUB_TOKEN, then run this cell again.\n\n"
            "  Kaggle grants a secret one notebook at a time. Attaching it in\n"
            "  another notebook does not attach it here.\n\n" + detail
        )
    return target
ARTEFACTS = {
    "database":    ("data/warehouse/eicu.db",                   None),
    "value index": ("data/warehouse/value_index.db",            None),
    "GLiNER2":     ("models/gliner2-base-v1",                   "model.safetensors"),
    "Qwen3-1.7B":  ("models/qwen3-1.7b/Qwen3-1.7B-Q4_K_M.gguf", None),
}
DEPTHS = ("", "*/", "*/*/", "*/*/*/", "*/*/*/*/", "*/*/*/*/*/")


def whole(path, probe=None):
    """Present and finished. A model directory with no weights in it is neither."""
    return (path / probe).exists() if probe else path.exists()


def find_input(relative, probe=None):
    """The first attached input carrying `relative`, at whatever depth it sits."""
    if not INPUTS.exists():
        return None
    for prefix in DEPTHS:
        for hit in sorted(INPUTS.glob(prefix + relative)):
            if whole(hit, probe):
                return hit
    return None


def attached_inputs():
    """The inputs actually attached, named by what they carry rather than by the
    directory level Kaggle happens to mount them under."""
    if not INPUTS.exists():
        return []
    markers = ("src", "data", "models", "nl2sql")
    return [p.relative_to(INPUTS).as_posix()
            for pattern in ("*", "*/*", "*/*/*")
            for p in sorted(INPUTS.glob(pattern))
            if p.is_dir() and any((p / m).exists() for m in markers)]


def locate(project):
    """Every artefact, in the working copy or in an attached input."""
    found, missing = {}, []
    for label, (relative, probe) in ARTEFACTS.items():
        if label == "value index":
            continue                       # always beside the database, see below
        local = project / relative
        path = local if whole(local, probe) else find_input(relative, probe)
        (found.__setitem__(label, path) if path else missing.append(label))

    # The package derives the index path from the database path, so the two must
    # be in the same directory. Looking for it anywhere else would resolve here
    # and fail there.
    if "database" in found:
        index = found["database"].with_name("value_index.db")
        found["value index"] = index if index.exists() else missing.append("value index")
    else:
        missing.append("value index")
    return found, [m for m in missing if m]


def configure(found):
    os.environ["DB_PATH"]             = str(found["database"])
    os.environ["GLINER_MODEL"]        = str(found["GLiNER2"])
    os.environ["LOCAL_LLM_GGUF_PATH"] = str(found["Qwen3-1.7B"])
    os.environ["LOCAL_LLM_THREADS"]   = str(max(2, os.cpu_count() or 4))
    os.environ["LOCAL_LLM_BACKEND"]   = "llamacpp"
    os.environ["PRIVACY_MODE"]        = "demo"
    os.environ["LANGSMITH_PROJECT"]   = "nl2sql"
    os.environ["LANGSMITH_TRACING"]   = "1" if os.environ.get("LANGSMITH_API_KEY") else "0"


def size_mb(path):
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return path.stat().st_size / 1e6 if path.exists() else 0.0


def show(found):
    for label, (relative, _) in ARTEFACTS.items():
        path = found.get(label)
        if path is None:
            print(f"  {label:<14}{'missing':>10}")
            continue
        root = path.parents[len(Path(relative).parts) - 1]
        if WORK in path.parents:
            where = "built here"
        elif INPUTS.exists() and (INPUTS in root.parents or root == INPUTS):
            where = root.relative_to(INPUTS).as_posix()
        else:
            where = str(root)
        print(f"  {label:<14}{size_mb(path):>9.0f} MB   {where}")
print("secrets")
load_secrets()

print("\ncode")
PROJECT = get_code()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print(f"  {PROJECT}")

configure({label: PROJECT / relative for label, (relative, _) in ARTEFACTS.items()})

In [ ]:
commit = (PROJECT / "COMMIT").read_text().strip()[:8] if (PROJECT / "COMMIT").exists() else ""
if not commit:
    commit = subprocess.run(["git", "-C", str(PROJECT), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip() or "unknown"
modules = sorted(p.name for p in (PROJECT / "src/hybridsql").iterdir() if p.is_dir())
print(f"  commit    {commit}")
print(f"  packages  {', '.join(m for m in modules if not m.startswith('_'))}")

---

## 3. Dependencies

`llama-cpp-python` compiles from source, which takes 5 to 15 minutes. The
prebuilt CPU wheel index turns that into a 20-second download.

In [ ]:
%%capture --no-stderr
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    "llama-cpp-python>=0.3" "gliner2>=1.3" "langgraph>=1.0" "langsmith>=0.10" \
    "fastapi>=0.115" "uvicorn[standard]>=0.34" "pydantic-settings>=2.6" \
    "sqlglot>=25.0" "rapidfuzz>=3.10" "pyyaml>=6.0" "httpx>=0.27" "python-dotenv>=1.0"

In [ ]:
import importlib

for name in ["gliner2", "llama_cpp", "langgraph", "langsmith", "fastapi", "sqlglot", "rapidfuzz"]:
    module = importlib.import_module(name)
    print(f"  {name:<14}{getattr(module, '__version__', 'installed')}")

---

## 4. The models

Both models run **inside this process**. Neither is an API call, which is what
makes "local" something you can check rather than something claimed.

**GLiNER2** reads the question and returns the entities in it. It is zero-shot:
the entity types are described in plain English at call time, so adding a type
costs a line of text rather than a training run.

**Qwen3-1.7B Q4** writes the final answer from the rows the query returned. It is
deliberately small. The SQL came from a much larger model and the figures come
from the database, so this model only has to turn numbers into a sentence.

They are fetched with `curl -C -`, which resumes a broken transfer at the real
byte. `huggingface_hub` cannot: its temporary filename derives from the signed
CDN URL, which changes on every attempt, so each retry restarts from zero.

In [ ]:
started = time.time()
!bash scripts/download_models.sh
print(f"\n{time.time() - started:.0f}s")

---

## 5. The database

The published eICU export has **no primary keys, no foreign keys and no
indexes**. That matters here more than it would elsewhere, because the schema is
the only thing the cloud model ever sees of this database. A model told how the
tables join writes a correct query. A model left to guess guesses.

So the build reconstructs the schema:

- **Primary keys and indexes** come from the consortium's own repository
  (`MIT-LCP/eicu-code`), which defines 17 keys and 22 indexes.
- **Foreign keys** are declared here. eICU documents the relationships but
  declares none: `patientunitstayid` links 28 tables to `patient`, `hospitalid`
  links `patient` to `hospital`.
- **Composite indexes** on "join key + filtered column" for the seven tables that
  questions actually filter. Without them the rebuilt database was slower than
  the raw export. With them SQLite uses a covering index.

In [ ]:
started = time.time()
!python scripts/build_database.py
print(f"\n{time.time() - started:.0f}s")

In [ ]:
import sqlite3

db = sqlite3.connect(os.environ["DB_PATH"])
tables = [r[0] for r in db.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
rows       = sum(db.execute('SELECT COUNT(*) FROM "%s"' % t).fetchone()[0] for t in tables)
keys       = sum(1 for t in tables if any(c[5] for c in db.execute('PRAGMA table_info("%s")' % t)))
foreign    = sum(len(db.execute('PRAGMA foreign_key_list("%s")' % t).fetchall()) for t in tables)
violations = db.execute("PRAGMA foreign_key_check").fetchall()
db.close()

print(f"  tables         {len(tables)}")
print(f"  rows           {rows:,}")
print(f"  primary keys   {keys}")
print(f"  foreign keys   {foreign}")
print(f"  size           {Path(os.environ['DB_PATH']).stat().st_size / 1e6:.0f} MB")
print(f"  violations     {len(violations)}")

assert not violations, "the declared foreign keys do not hold"

---

## 6. The value index

### The problem it solves

A question says *aspirin*. The database stores `ASPIRIN EC 81 MG PO TBEC`. So
`WHERE drugname = 'aspirin'` returns **nothing**, and the question fails without
anyone noticing.

The index is a small second database that maps what a person writes to what is
actually stored. It is built **here, once**, and it is what makes masking possible
at all: it is the moment the system learns which exact string must never leave.

### Why not simply index every value

Because the cost would then grow with the number of rows, and a design that stops
working on a large database is not a design. So every text column is measured once
and put in one of three tiers.

| Tier | What the column holds | What is stored | Example |
|---|---|---|---|
| **A** | a bounded vocabulary | **every distinct value** | `medication.drugname`, 1 402 names |
| **B** | thousands of distinct strings | **nothing** — searched on demand | `lab.labresulttext`, 7 669 |
| **C** | identifiers, timestamps, free text | **nothing** — never searched | `patientunitstayid` |

The decision is made from measurements, not from column names: how many distinct
values, how long they are, and how close the distinct count is to the row count. A
column with almost as many distinct values as rows is an identifier whatever it is
called — that is how `patient.uniquepid`, 73 % unique, was kept out of a file meant
for business vocabulary.

**What this buys.** The stored size is `columns x limit` — it does not depend on the
number of rows. Adding ten million rows changes nothing. If a column outgrows the
limit it moves to tier B, which stores *less*. The index below stays a few megabytes
whatever this database grows into.

Notebook 2 shows a lookup running, and what happens when the scorer gets it wrong.

In [ ]:
started = time.time()
!python scripts/build_value_index.py
print(f"\n{time.time() - started:.0f}s")

---

## 7. Check, then save

`data/raw` holds the archive and the export the database was built from. Neither
is read again once `eicu.db` exists, and leaving them in place would add half a
gigabyte to every notebook that attaches this output. They are removed first.

Then **Save Version > Save & Run All**.

In [ ]:
from hybridsql.db import value_index

raw = PROJECT / "data/raw"
if raw.exists() and Path(os.environ["DB_PATH"]).exists():
    freed = sum(f.stat().st_size for f in raw.rglob("*") if f.is_file()) / 1e6
    shutil.rmtree(raw)
    print(f"  removed the build cache, {freed:.0f} MB\n")

FINAL = {label: PROJECT / relative for label, (relative, _) in ARTEFACTS.items()}
show(FINAL)

incomplete = [label for label, (relative, probe) in ARTEFACTS.items()
              if not whole(PROJECT / relative, probe)]
assert not incomplete, f"not built: {incomplete}"

s = value_index.stats()
print(f"\n  {s['values_indexed']:,} values indexed over {s['tiers']['A']} columns")
print(f"  {sum(size_mb(p) for p in FINAL.values()):.0f} MB will be saved as this notebook's output")

---

## 8. The files a question passes through

In order. Everything else in the repository builds these, measures them, or
serves them. The line counts are read from the files that were just cloned, so
this list cannot drift away from the code.

If you only read one, read `security/egress_gate.py`.

In [ ]:
PIPELINE = [
 ("pipeline/understand.py",   "reads the question and picks out the words that matter"),
 ("providers/extractor.py",   "the small model that finds those words"),
 ("resources/glossary.py",    "knows that 'drug' means the drugname column"),
 ("db/value_index.py",        "turns 'aspirin' into the exact value stored in the database"),
 ("db/catalog.py",           "tells a value apart from a column name the analyst typed"),
 ("pipeline/anonymize.py",    "swaps every value for a symbol, :v1 and :v2"),
 ("pipeline/opaque.py",       "hides the table and column names as well"),
 ("db/schema.py",             "describes the tables to the cloud model"),
 ("pipeline/generate.py",     "writes the prompt and asks the cloud model for SQL"),
 ("security/egress_gate.py",  "checks every word before anything is sent"),
 ("providers/cloud.py",       "the only file that opens a connection to the internet"),
 ("security/sql_validator.py","refuses anything that is not a plain SELECT"),
 ("db/connection.py",         "runs the query, read-only"),
 ("providers/local_model.py", "the small model that writes the final answer"),
 ("pipeline/answer.py",       "turns the rows into a sentence"),
 ("graph/build.py",           "wires these stages into the four architectures"),
]

print(f"  {'file':<28}{'lines':>6}   what it does")
total = 0
for name, purpose in PIPELINE:
    path = PROJECT / "src/hybridsql" / name
    n = path.read_text(encoding="utf-8", errors="replace").count(chr(10))
    total += n
    print(f"  {name:<28}{n:>6}   {purpose}")

print(f"\n  {total:,} lines")

---

Save the version now. The other three notebooks read what it contains.

**Next:** [2. Understanding](https://www.kaggle.com/code/kirazul/nl2sql-2-understanding)